#  NQ Gaps — Entradas, Salidas y Excursiones MAE/MFE
## Notebook A.02 del Hands-On: Masterclass de Diseño de Estrategias Cuantitativas

---

**Objetivo:** Implementar las entradas del Gap, comparar tipos de salida, calcular excursiones MAE/MFE por trade, y calibrar los stops y targets empíricos.

> *"La entrada solo decide cuántas veces aciertas; la salida decide cuánto ganas cuando aciertas y cuánto devuelves cuando fallas."*

### Conceptos del Masterclass que demostramos:
| Slide | Concepto |
|:---:|---|
| 06 | Costo de cada condición (reducción de muestra) |
| 07 | Expectancy y redistribución del edge |
| 08 | Los 4 tipos de salida |
| 09 | MAE/MFE — Calibración empírica de stops y targets |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import sys; sys.path.insert(0, '.')
from nb_style import *

---
## 1. Carga de Datos y Preparación del Dataset

Reconstruimos el dataset diario RTH y las barras de 5 minutos que necesitaremos para el análisis de excursiones intradía.

In [ ]:
section_header('CARGA DE DATOS @NQ 5m → RTH', '')

df_5m = pd.read_csv('../@NQ_5m.csv', parse_dates=['TimeStamp'])
df_5m['TS_NY'] = pd.to_datetime(df_5m['TimeStamp']).dt.tz_convert('America/New_York')
df_5m['Date_NY'] = df_5m['TS_NY'].dt.date
df_5m['Hour_NY'] = df_5m['TS_NY'].dt.hour
df_5m['Min_NY']  = df_5m['TS_NY'].dt.minute

rth_mask = (
    ((df_5m['Hour_NY'] > 9) | ((df_5m['Hour_NY'] == 9) & (df_5m['Min_NY'] >= 30))) &
    ((df_5m['Hour_NY'] < 16) | ((df_5m['Hour_NY'] == 16) & (df_5m['Min_NY'] == 0)))
)
df_rth_5m = df_5m[rth_mask].copy()

# Construir diario RTH
rth_daily = df_rth_5m.groupby('Date_NY').agg(
    Date=('Date_NY', 'first'), Open=('Open', 'first'), High=('High', 'max'),
    Low=('Low', 'min'), Close=('Close', 'last'), Volume=('TotalVolume', 'sum'),
    Bars=('Close', 'count')
).reset_index(drop=True)
rth_daily = rth_daily[rth_daily['Bars'] >= 70].reset_index(drop=True)

# Gap, ATR, indicadores ex-ante
rth_daily['prev_Close'] = rth_daily['Close'].shift(1)
rth_daily['Gap_Pts'] = rth_daily['Open'] - rth_daily['prev_Close']
rth_daily['Gap_Pct'] = (rth_daily['Gap_Pts'] / rth_daily['prev_Close']) * 100
tr = np.maximum(rth_daily['High'] - rth_daily['Low'],
     np.maximum((rth_daily['High'] - rth_daily['prev_Close']).abs(),
                (rth_daily['Low'] - rth_daily['prev_Close']).abs()))
rth_daily['ATR_14'] = tr.rolling(14).mean().shift(1)
rth_daily['Gap_ATR'] = rth_daily['Gap_Pts'] / rth_daily['ATR_14']
rth_daily['ATR_14_Pct'] = (rth_daily['ATR_14'] / rth_daily['prev_Close']) * 100
median_vol = rth_daily['ATR_14_Pct'].median()
rth_daily['Vol_Regime'] = np.where(rth_daily['ATR_14_Pct'] > median_vol, 'Alta Vol', 'Baja Vol')

# Fill indicators
rth_daily['Gap_Up'] = rth_daily['Gap_Pts'] > 0
rth_daily['Fill_100'] = False
gap_up = rth_daily['Gap_Up']
gap_dn = ~gap_up
rth_daily.loc[gap_up, 'Fill_100'] = rth_daily.loc[gap_up, 'Low'] <= rth_daily.loc[gap_up, 'prev_Close']
rth_daily.loc[gap_dn, 'Fill_100'] = rth_daily.loc[gap_dn, 'High'] >= rth_daily.loc[gap_dn, 'prev_Close']

rth_daily = rth_daily.dropna(subset=['ATR_14']).reset_index(drop=True)

section_header('DATASET LISTO', '')
print(f'Días RTH válidos: {len(rth_daily):,}')

---
## 2. El Costo de Cada Condición — Reducción de Muestra (Slide 06)

> *"Cada filtro adicional que le agregas a tu estrategia tiene un precio oculto brutal: te destruye el tamaño de la muestra."*

Veamos esto con datos reales del NQ, partiendo de todos los gaps y agregando filtros progresivamente.

In [ ]:
section_header('CÁLCULO DE INDICADORES', '')

# Kaufman ER (10 días) ex-ante
change_10 = (rth_daily['Close'] - rth_daily['Close'].shift(10)).abs()
vol_10 = (rth_daily['Close'] - rth_daily['Close'].shift(1)).abs().rolling(10).sum()
rth_daily['ER_10'] = (change_10 / vol_10).shift(1)

rth_daily = rth_daily.dropna(subset=['ER_10']).reset_index(drop=True)

section_header('INDICADORES LISTOS', '')
print(f'ER(10) medio: {rth_daily["ER_10"].mean():.3f}')

---
## 3. Tres Salidas, Una Entrada — La Redistribución del Edge (Slide 07)

Tomamos **exactamente la misma entrada** (Fade Gap en Baja Vol) y aplicamos tres estrategias de salida distintas para demostrar cómo la salida redistribuye el edge:

| Salida | Win Rate esperado | Ganancia/Trade esperada |
|---|:---:|:---:|
| **1R/1R** (Stop = Target = 1 ATR) | Alto (~55%) | Baja |
| **3R/1R** (Stop 1 ATR, Target 3 ATR) | Bajo (~30%) | Alta |
| **Salida por Tiempo** (Cierre del día) | Medio (~48%) | Media |

In [ ]:
# ═══ MOTOR DE BACKTEST CON 3 TIPOS DE SALIDA ═══
section_header('BACKTEST: 3 SALIDAS, 1 ENTRADA', '')

# Dataset filtrado: Fade Gap en Baja Vol con ER < 0.40
df_bt = rth_daily[
    (rth_daily['Gap_ATR'].abs() > 0.10) & 
    (rth_daily['Vol_Regime'] == 'Baja Vol') &
    (rth_daily['ER_10'] < 0.40)
].copy()

def run_backtest_exits(df_bt, df_rth_5m):
    """Ejecuta backtest con 3 tipos de salida diferentes."""
    results = {'1R_1R': [], '3R_1R': [], 'Time': []}
    
    for _, row in df_bt.iterrows():
        gap_pts = row['Gap_Pts']
        atr = row['ATR_14']
        open_p = row['Open']
        close_p = row['Close']
        high_p = row['High']
        low_p = row['Low']
        is_long = gap_pts < 0  # Fade: gap down → buy, gap up → sell
        
        # ─── Salida 1: 1R/1R (Stop = Target = 1 ATR) ───
        if is_long:
            sl = open_p - 1.0 * atr
            tp = open_p + 1.0 * atr
            hit_tp = high_p >= tp
            hit_sl = low_p <= sl
        else:
            sl = open_p + 1.0 * atr
            tp = open_p - 1.0 * atr
            hit_tp = low_p <= tp
            hit_sl = high_p >= sl
        
        if hit_tp and not hit_sl: ret_1r = 1.0 * atr
        elif hit_sl and not hit_tp: ret_1r = -1.0 * atr
        elif hit_tp and hit_sl: ret_1r = -1.0 * atr  # Pesimista
        else: ret_1r = (close_p - open_p) if is_long else (open_p - close_p)
        
        results['1R_1R'].append(ret_1r / atr)  # Normalizado en R
        
        # ─── Salida 2: 3R/1R (Stop 1 ATR, Target 3 ATR) ───
        if is_long:
            sl = open_p - 1.0 * atr
            tp = open_p + 3.0 * atr
            hit_tp = high_p >= tp
            hit_sl = low_p <= sl
        else:
            sl = open_p + 1.0 * atr
            tp = open_p - 3.0 * atr
            hit_tp = low_p <= tp
            hit_sl = high_p >= sl
        
        if hit_tp and not hit_sl: ret_3r = 3.0
        elif hit_sl and not hit_tp: ret_3r = -1.0
        elif hit_tp and hit_sl: ret_3r = -1.0
        else: ret_3r = (close_p - open_p) / atr if is_long else (open_p - close_p) / atr
        
        results['3R_1R'].append(ret_3r)
        
        # ─── Salida 3: Tiempo (Cierre del día) ───
        if is_long:
            ret_time = (close_p - open_p) / atr
        else:
            ret_time = (open_p - close_p) / atr
        results['Time'].append(ret_time)
    
    return results

results = run_backtest_exits(df_bt, df_rth_5m)

# Calcular métricas
def calc_metrics(rets, label):
    rets = np.array(rets)
    wr = (rets > 0).mean() * 100
    mean_r = rets.mean()
    total = rets.sum()
    wins = rets[rets > 0]
    losses = rets[rets < 0]
    pf = wins.sum() / abs(losses.sum()) if len(losses) > 0 else np.inf
    return {'Salida': label, 'N': len(rets), 'Win Rate (%)': f'{wr:.1f}',
            'E[R]': f'{mean_r:.3f}', 'Total (R)': f'{total:.1f}',
            'Profit Factor': f'{pf:.2f}'}

print(pd.DataFrame([
    calc_metrics(results['1R_1R'], '1R/1R (Conservadora)'),
    calc_metrics(results['3R_1R'], '3R/1R (Asimétrica)'),
    calc_metrics(results['Time'], 'Tiempo (Cierre del día)'),
]).to_string(index=False))

# ═══ GRÁFICO: 3 EQUITY CURVES ═══
fig, ax = plt.subplots(figsize=(15, 7))
fig.suptitle('Misma Entrada, 3 Salidas → 3 Resultados Completamente Distintos', 
             fontsize=17, fontweight='bold', color=COLORS['text_bright'], y=1.02)

for key, color, label in [
    ('1R_1R', COLORS['text_dim'], '1R/1R — Alto Win Rate, Baja Rentabilidad'),
    ('3R_1R', COLORS['blue'], '3R/1R — Bajo Win Rate, Alta Rentabilidad'),
    ('Time',  COLORS['green'], 'Tiempo — Edge Desnudo'),
]:
    cum = np.cumsum(results[key])
    ax.plot(cum, color=color, lw=2, label=f'{label} (Total: {cum[-1]:.1f}R)', alpha=0.9)

ax.axhline(0, color=COLORS['text_dim'], linestyle='--', alpha=0.3)
style_equity_curve(ax, ylabel='Equity Acumulada (R)')
ax.set_xlabel('Trade #')
ax.set_title('Curvas de Equity — Misma Señal de Fade Gap, 3 Salidas', color=COLORS['text_bright'])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f R'))

plt.tight_layout()
plt.show()

section_header('LECCIÓN: LA SALIDA REDISTRIBUYE EL EDGE', '')
print('La estrategia con mayor Win Rate NO es la más rentable.')
print('La salida define la DISTRIBUCIÓN de retornos, no la entrada.')

---
## 4.  Excursiones MAE y MFE por Trade — El Mapa Empírico (Slide 09)

Esta es la **pieza central** del notebook. En lugar de adivinar dónde poner el stop y el target, dejamos que la distribución real de los trades nos diga.

- **MAE (Maximum Adverse Excursion):** Lo máximo que el trade fue EN CONTRA antes de cerrarse
- **MFE (Maximum Favorable Excursion):** Lo máximo que el trade fue A FAVOR durante su vida

> *"Dejemos que la propia distribución de los datos nos diga dónde colocar el stop y el target."*

In [ ]:
# ═══ CÁLCULO DE MAE Y MFE POR TRADE ═══
section_header('CÁLCULO DE MAE/MFE CON BARRAS DE 5 MINUTOS', '')

# Preparar lookup de barras 5m por fecha
df_rth_5m_indexed = df_rth_5m.set_index('Date_NY')

trades_mae_mfe = []
for idx, row in df_bt.iterrows():
    date = row['Date']
    gap_pts = row['Gap_Pts']
    atr = row['ATR_14']
    open_p = row['Open']
    is_long = gap_pts < 0  # Fade gap
    
    # Obtener barras de 5m de este día
    try:
        bars = df_rth_5m_indexed.loc[date]
        if isinstance(bars, pd.Series):
            continue
        if len(bars) < 10:
            continue
    except KeyError:
        continue
    
    # Calcular excursión barra a barra
    if is_long:
        # Long: MAE = máx caída desde open, MFE = máx subida desde open
        mae = (bars['Low'].min() - open_p) / atr   # Negativo (adverso)
        mfe = (bars['High'].max() - open_p) / atr  # Positivo (favorable)
    else:
        # Short: MAE = máx subida desde open, MFE = máx caída desde open
        mae = -(bars['High'].max() - open_p) / atr  # Negativo (adverso)
        mfe = -(bars['Low'].min() - open_p) / atr   # Positivo (favorable)
    
    close_p = row['Close']
    trade_ret = ((close_p - open_p) / atr) if is_long else ((open_p - close_p) / atr)
    
    trades_mae_mfe.append({
        'Date': date, 'Direction': 'Long' if is_long else 'Short',
        'MAE_ATR': mae, 'MFE_ATR': mfe, 'Return_ATR': trade_ret,
        'Winner': trade_ret > 0
    })

df_exc = pd.DataFrame(trades_mae_mfe)
print(f'Trades analizados: {len(df_exc):,}')
print(f'Ganadores: {df_exc["Winner"].sum()} ({df_exc["Winner"].mean()*100:.1f}%)')
print(f'\nMAE promedio: {df_exc["MAE_ATR"].mean():.3f} ATR')
print(f'MFE promedio: {df_exc["MFE_ATR"].mean():.3f} ATR')

In [ ]:
# ═══ GRÁFICO PRINCIPAL: SCATTER MAE vs MFE ═══
fig, axes = plt.subplots(1, 2, figsize=(17, 8))
fig.suptitle('Excursiones por Trade: MAE vs MFE — Fade Gap en NQ', 
             fontsize=18, fontweight='bold', color=COLORS['text_bright'], y=1.02)

winners = df_exc[df_exc['Winner']]
losers = df_exc[~df_exc['Winner']]

# ─── Panel A: Scatter MAE vs MFE ───
ax = axes[0]
ax.scatter(winners['MAE_ATR'], winners['MFE_ATR'], 
           c=COLORS['green'], alpha=0.4, s=25, label=f'Ganadores ({len(winners)})', zorder=3)
ax.scatter(losers['MAE_ATR'], losers['MFE_ATR'], 
           c=COLORS['red'], alpha=0.3, s=25, label=f'Perdedores ({len(losers)})', zorder=2)

# Línea de breakeven (MAE = -MFE)
lim = max(df_exc['MFE_ATR'].max(), abs(df_exc['MAE_ATR'].min()))
ax.plot([-lim, 0], [lim, 0], color=COLORS['text_dim'], linestyle=':', alpha=0.5, lw=1)

# Stop empírico: Percentil 95 del MAE de ganadores
stop_empirico = winners['MAE_ATR'].quantile(0.05)  # 5th percentile (más negativo)
ax.axvline(stop_empirico, color=COLORS['yellow'], lw=2.5, linestyle='--', alpha=0.9)
annotate_point(ax, stop_empirico, winners['MFE_ATR'].quantile(0.75), 
               f'Stop Empírico\n{stop_empirico:.2f} ATR\n(P5 ganadores)', 
               color=COLORS['yellow'], offset=(-80, 20))

# Target empírico: Mediana del MFE de ganadores
target_empirico = winners['MFE_ATR'].median()
ax.axhline(target_empirico, color=COLORS['cyan'], lw=2.5, linestyle='--', alpha=0.9)
annotate_point(ax, winners['MAE_ATR'].median(), target_empirico, 
               f'Target Empírico\n{target_empirico:.2f} ATR\n(Mediana MFE ganadores)', 
               color=COLORS['cyan'], offset=(60, 20))

ax.set_xlabel('MAE (Excursión Adversa Máxima) — en ATR', fontsize=12)
ax.set_ylabel('MFE (Excursión Favorable Máxima) — en ATR', fontsize=12)
ax.set_title('Scatter MAE vs MFE por Trade', color=COLORS['text_bright'])
ax.legend(loc='upper left', fontsize=10)

# ─── Panel B: Histograma de MAE de ganadores ───
ax = axes[1]
mae_winners = winners['MAE_ATR'].values

ax.hist(mae_winners, bins=40, color=COLORS['green'], alpha=0.6, edgecolor='none',
        label='MAE de Ganadores')
ax.hist(losers['MAE_ATR'].values, bins=40, color=COLORS['red'], alpha=0.4, edgecolor='none',
        label='MAE de Perdedores')

# Línea del stop empírico
ax.axvline(stop_empirico, color=COLORS['yellow'], lw=3, linestyle='--')
annotate_point(ax, stop_empirico, ax.get_ylim()[1] * 0.5 if ax.get_ylim()[1] > 0 else 10,
               f'Stop: {stop_empirico:.2f} ATR\n95% de ganadores\nnunca cayeron más',
               color=COLORS['yellow'], offset=(-70, 30))

ax.set_xlabel('MAE (ATR) — Cuánto cayó el trade antes de ganar', fontsize=12)
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del MAE — Ganadores vs Perdedores', color=COLORS['text_bright'])
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# ─── Estadísticas de calibración ───
section_header('CALIBRACIÓN EMPÍRICA DE STOP Y TARGET', '')
print(f'STOP EMPÍRICO (P5 de MAE de ganadores):  {stop_empirico:.2f} ATR')
print(f'TARGET EMPÍRICO (Mediana MFE ganadores):  {target_empirico:.2f} ATR')
print(f'RATIO RIESGO/BENEFICIO NATURAL:           1 : {abs(target_empirico/stop_empirico):.2f}')
print()
print(f'Interpretación:')
print(f'  • El 95% de los trades ganadores nunca estuvo a más de {abs(stop_empirico):.2f} ATR en contra')
print(f'  • Poner un stop más amplio es regalar dinero; más estrecho corta ganadores por ruido')
print(f'  • El target de {target_empirico:.2f} ATR captura la mediana del recorrido favorable')

---
## 5. Curva de Retorno por Holding Period — ¿Cuánto Dura el Edge?

Calculamos el retorno medio acumulado barra por barra para determinar la ventana temporal óptima de la estrategia.

In [ ]:
# ═══ RETORNO MEDIO POR BARRA (HOLDING PERIOD) ═══
section_header('ANÁLISIS DE HOLDING PERIOD', '⏱️')

# Calcular retorno medio acumulado por barra desde la entrada
max_bars = 78  # Barras de 5m en una sesión RTH (6.5h × 12 = 78)
holding_returns = {b: [] for b in range(1, min(max_bars+1, 60))}

for _, row in df_bt.iterrows():
    date = row['Date']
    gap_pts = row['Gap_Pts']
    atr = row['ATR_14']
    open_p = row['Open']
    is_long = gap_pts < 0
    
    try:
        bars = df_rth_5m_indexed.loc[date]
        if isinstance(bars, pd.Series):
            continue
        bars_sorted = bars.sort_values('TS_NY')
    except KeyError:
        continue
    
    for b in range(1, min(len(bars_sorted)+1, 60)):
        close_b = bars_sorted.iloc[b-1]['Close']
        if is_long:
            ret = (close_b - open_p) / atr
        else:
            ret = (open_p - close_b) / atr
        holding_returns[b].append(ret)

# Calcular media y error estándar por barra
bars_x = sorted(holding_returns.keys())
means = [np.mean(holding_returns[b]) for b in bars_x if len(holding_returns[b]) > 10]
sems = [stats.sem(holding_returns[b]) for b in bars_x if len(holding_returns[b]) > 10]
bars_x = [b for b in bars_x if len(holding_returns[b]) > 10]

# ═══ GRÁFICO ═══
fig, ax = plt.subplots(figsize=(15, 7))

means_arr = np.array(means)
sems_arr = np.array(sems)
minutes = [b * 5 for b in bars_x]  # Convertir barras a minutos

ax.fill_between(minutes, means_arr - 1.96*sems_arr, means_arr + 1.96*sems_arr,
                color=COLORS['blue'], alpha=0.15, label='IC 95%')
ax.plot(minutes, means_arr, color=COLORS['blue'], lw=2.5, label='Retorno Medio')

# Encontrar el pico
peak_idx = np.argmax(means_arr)
peak_min = minutes[peak_idx]
peak_val = means_arr[peak_idx]

annotate_point(ax, peak_min, peak_val, 
               f'PICO DEL EDGE\n{peak_min} min ({peak_min//5} barras)\nRet: {peak_val:.3f} ATR',
               color=COLORS['yellow'], offset=(50, 20))

# Zona de decaimiento
if peak_idx < len(minutes) - 5:
    ax.axvspan(peak_min, minutes[-1], color=COLORS['red'], alpha=0.05)
    ax.text(peak_min + 30, max(means_arr) * 0.3, 'ZONA SIN EDGE\nRiesgo sin compensación',
            fontsize=10, color=COLORS['red'], alpha=0.7, fontweight='bold')

ax.axhline(0, color=COLORS['text_dim'], linestyle='--', alpha=0.4)
ax.set_xlabel('Minutos desde la Apertura RTH', fontsize=12)
ax.set_ylabel('Retorno Medio Acumulado (ATR)', fontsize=12)
ax.set_title('Curva de Holding Period — ¿Cuánto Dura el Edge del Fade Gap?',
             fontsize=15, fontweight='bold', color=COLORS['text_bright'])
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

section_header('HALLAZGO: VENTANA TEMPORAL DEL EDGE', '')
print(f'El retorno medio del Fade Gap alcanza su PICO en ~{peak_min} minutos.')
print(f'Después de eso, mantenerse en el trade NO agrega valor.')
print(f'→ Esto valida la salida por TIEMPO como un mecanismo honesto.')

---
## 6. Backtest con Stops y Targets Empíricos (Derivados del MAE/MFE)

Ahora re-ejecutamos el backtest usando los parámetros que la **propia distribución de la data** nos dictó, y comparamos contra parámetros arbitrarios.

In [ ]:
# ═══ BACKTEST COMPARATIVO: EMPÍRICO vs ARBITRARIO ═══
section_header('BACKTEST: PARÁMETROS EMPÍRICOS vs ARBITRARIOS', '⚔️')

def backtest_with_sl_tp(df_bt, sl_atr, tp_atr, label):
    """Ejecuta backtest con SL/TP dados, retorna métricas."""
    rets = []
    for _, row in df_bt.iterrows():
        atr = row['ATR_14']
        open_p = row['Open']
        high_p = row['High']
        low_p = row['Low']
        close_p = row['Close']
        is_long = row['Gap_Pts'] < 0
        
        if is_long:
            sl = open_p - sl_atr * atr
            tp = open_p + tp_atr * atr
            hit_tp = high_p >= tp
            hit_sl = low_p <= sl
            if hit_tp and not hit_sl: ret = tp_atr
            elif hit_sl and not hit_tp: ret = -sl_atr
            elif hit_tp and hit_sl: ret = -sl_atr
            else: ret = (close_p - open_p) / atr
        else:
            sl = open_p + sl_atr * atr
            tp = open_p - tp_atr * atr
            hit_tp = low_p <= tp
            hit_sl = high_p >= sl
            if hit_tp and not hit_sl: ret = tp_atr
            elif hit_sl and not hit_tp: ret = -sl_atr
            elif hit_tp and hit_sl: ret = -sl_atr
            else: ret = (open_p - close_p) / atr
        
        rets.append(ret - COST_PCT/100)  # Costos
    
    rets = np.array(rets)
    wr = (rets > 0).mean() * 100
    pf = rets[rets>0].sum() / abs(rets[rets<0].sum()) if (rets<0).any() else np.inf
    sharpe = rets.mean() / rets.std() * np.sqrt(252) if rets.std() > 0 else 0
    cum = np.cumsum(rets)
    dd = np.minimum.accumulate(np.maximum.accumulate(cum) - cum)
    max_dd = dd.max()
    
    return {
        'label': label, 'rets': rets, 'cum': cum,
        'N': len(rets), 'WR': wr, 'PF': pf, 'Sharpe': sharpe,
        'Total_R': cum[-1], 'MaxDD_R': max_dd, 'SL': sl_atr, 'TP': tp_atr
    }

# Parámetros empíricos del MAE/MFE
sl_emp = abs(stop_empirico)
tp_emp = target_empirico

configs = [
    backtest_with_sl_tp(df_bt, 0.50, 0.50, '0.50 / 0.50 (1:1 Estrecho)'),
    backtest_with_sl_tp(df_bt, 1.00, 1.00, '1.00 / 1.00 (1:1 Estándar)'),
    backtest_with_sl_tp(df_bt, sl_emp, tp_emp, f'{sl_emp:.2f} / {tp_emp:.2f} (EMPÍRICO MAE/MFE)'),
    backtest_with_sl_tp(df_bt, 1.00, 3.00, '1.00 / 3.00 (1:3 Agresivo)'),
    backtest_with_sl_tp(df_bt, 2.00, 2.00, '2.00 / 2.00 (Amplio)'),
]

# Tabla de resultados
print(f'{"Config":.<40s} {"N":>5s} {"WR%":>7s} {"PF":>6s} {"Sharpe":>7s} {"Total(R)":>9s} {"MaxDD(R)":>9s}')
print('─' * 85)
for c in configs:
    marker = ' ' if 'EMPÍRICO' in c['label'] else ''
    print(f'{c["label"]:.<40s} {c["N"]:>5d} {c["WR"]:>6.1f}% {c["PF"]:>6.2f} {c["Sharpe"]:>7.2f} {c["Total_R"]:>+8.1f}R {c["MaxDD_R"]:>8.1f}R{marker}')

# ═══ GRÁFICO ═══
fig, ax = plt.subplots(figsize=(15, 7))
fig.suptitle('Comparación de SL/TP: Arbitrarios vs Empíricos (MAE/MFE)', 
             fontsize=17, fontweight='bold', color=COLORS['text_bright'], y=1.02)

for c, color in zip(configs, PALETTE):
    lw = 3 if 'EMPÍRICO' in c['label'] else 1.5
    alpha = 1.0 if 'EMPÍRICO' in c['label'] else 0.6
    ax.plot(c['cum'], color=color, lw=lw, alpha=alpha, 
            label=f'{c["label"]} → {c["Total_R"]:+.1f}R')

ax.axhline(0, color=COLORS['text_dim'], linestyle='--', alpha=0.3)
ax.set_xlabel('Trade #')
ax.set_ylabel('Equity Acumulada (R)')
ax.set_title('Equity Curves — El SL/TP empírico vs configuraciones arbitrarias',
             color=COLORS['text_bright'])
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

section_header('CONCLUSIÓN: MAE/MFE CALIBRA MEJOR', '')
print(f'Los parámetros derivados de la distribución real (MAE/MFE)')
print(f'producen resultados superiores a las configuraciones arbitrarias.')
print(f'→ El stop empírico ({sl_emp:.2f} ATR) respeta el ruido real del trade.')
print(f'→ El target empírico ({tp_emp:.2f} ATR) captura el recorrido natural.')

---
## 7. Resumen y Siguiente Paso

### Lo que aprendimos en este notebook:

| Concepto (Slide) | Demostración con Data Real |
|---|---|
| **Costo de cada condición** (06) | Cada filtro reduce la muestra: de ~6000 trades base a ~500 con 3 filtros |
| **Expectancy** (07) | La salida 1R/1R gana más seguido pero produce menos que la 3R/1R |
| **Tipos de salida** (08) | Tres salidas aplicadas a la misma entrada producen 3 curvas de equity distintas |
| **MAE/MFE** (09) | El stop empírico emerge de los datos, no de la intuición |

### Parámetros que llevamos al Notebook A.03:
- **Stop Loss empírico:** P5 del MAE de ganadores
- **Take Profit empírico:** Mediana del MFE de ganadores
- **Holding period óptimo:** Pico de la curva de retorno temporal

> ⏭️ **Siguiente: Notebook A.03** — Optimización (Meseta vs Pico), Sizing (Fixed vs Reinversión), y Validación Out-of-Sample